## 🎯 What you will learn:


✅ How to subset spatial data for a specific study area (Cairo)


✅ How to combine clipping and overlay to control spatial extent


✅ How to simplify road networks using dissolve before analysis


✅ How to perform buffer-based proximity analysis with spatial join


✅ How to interpret and visualize proximity results on a map

In [ ]:
# Import Libraries
import geopandas as gpd
import matplotlib.pyplot as plt
import os

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define output folder directory
output_folder = '/content/drive/MyDrive/2-GeoPandas Mini Course/5-Buffer/Outputs'

In [ ]:
# Datasets Paths
# administrative boundaries (ADM1) Shapefile
adm_path = '/content/drive/MyDrive/2-GeoPandas Mini Course/2-GeoPackage Data Handling, CRS & Cleaning/Data/egy_admin1.shp'

# Egypt OSM data GeoPackage
osm_path = '/content/drive/MyDrive/2-GeoPandas Mini Course/2-GeoPackage Data Handling, CRS & Cleaning/Data/egypt.gpkg'

###Load the datasets

In [ ]:
# Load ADM1 Data
adm = gpd.read_file(adm_path)
adm.head()

In [ ]:
# Filter Cairo only and select needed columns
cairo = adm[adm['adm1_name'] == 'Cairo'][['adm1_name' ,'geometry' ]].copy()
cairo

In [ ]:
# Load OSM Data
# Inspect GeoPackage Layers
osm_layers = gpd.list_layers(osm_path)
osm_layers

In [ ]:
# Load POIs, and Roads
poi = gpd.read_file(osm_path , layer='gis_osm_pois_free')
roads = gpd.read_file(osm_path , layer ='gis_osm_roads_free' )

In [ ]:
# Filter restaurants
restaurants = poi[poi['fclass'] == 'restaurant'][['fclass', 'geometry']].copy()
restaurants

In [ ]:
# Filter Primary Roads and keep only necessary columns
primary_roads = roads[roads['fclass'] == 'primary'][['fclass', 'geometry']].copy()

###Align CRS (projected for buffer)

In [ ]:
# Convert all layers to projected CRS (meters)
cairo = cairo.to_crs(epsg=32636)
restaurants = restaurants.to_crs(epsg=32636)
primary_roads = primary_roads.to_crs(epsg=32636)

###Clip roads to Cairo (Overlay)

In [ ]:
# Clip roads inside Cairo boundaries
roads_cairo = gpd.overlay(primary_roads, cairo, how='intersection')

roads_cairo

### Clip restaurants to Cairo
To ensure we only analyze restaurants within the city limits.

In [ ]:
# Clip restaurants to Cairo boundary
restaurants_cairo = gpd.clip(restaurants, cairo)

restaurants_cairo

### Create buffer (500m) around Primary Roads in Cairo

In [ ]:
#Create 500m buffer around roads
roads_buffer = roads_cairo.buffer(500)

# Convert into GeoDataFram
road_buffer = gpd.GeoDataFrame(geometry=roads_buffer , crs= roads_cairo.crs )

road_buffer

###Spatial Join (restaurants within buffer)

In [ ]:
# Find restaurants within buffer
restaurants_near = gpd.sjoin(restaurants_cairo, road_buffer, how='inner', predicate='within')

restaurants_near

### Dissolve overlapping buffers
To avoid counting the same restaurant multiple times for different road segments, we merge (dissolve) all buffer polygons into a single geometry layer.

In [ ]:
# Dissolve all road buffers into one single shape
road_buffer_dissolved = road_buffer.dissolve()

# run the spatial join on the dissolved layer using only Cairo restaurants
restaurants_near_clean = gpd.sjoin(restaurants_cairo, road_buffer_dissolved, how='inner', predicate='within')

restaurants_near_clean

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Plot 1: Individual Buffers (duplicates)
road_buffer.plot(ax=ax1, edgecolor='blue', facecolor='none', alpha=0.5)
ax1.set_title("Individual Road Buffers\n(overlapping boundaries)", fontsize=14)
ax1.set_axis_off()

# Plot 2: Dissolved Buffer (The clean version)
road_buffer_dissolved.plot(ax=ax2, color='royalblue', alpha=0.5)
ax2.set_title("Dissolved Road Buffer\n(Unified geometry, no overlaps)", fontsize=14)
ax2.set_axis_off()

plt.tight_layout()
plt.show()

print(f"Number of features in raw road_buffer: {len(road_buffer)}")
print(f"Number of features in dissolved road_buffer: {len(road_buffer_dissolved)}")

### Calculate Percentage of Features within Distance

In [ ]:
# Calculate the percentage of restaurants within 1km of primary roads
total_restaurants_count = len(restaurants_cairo)
near_restaurants_count = len(restaurants_near_clean)

percentage = (near_restaurants_count / total_restaurants_count) * 100

print(f"Total restaurants: {total_restaurants_count}")
print(f"Restaurants within 500m of primary roads: {near_restaurants_count}")
print(f"Percentage: {percentage:.2f}%")

### Visualize Results
Plotting Cairo, the road buffers, and the restaurants within them.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines


fig, ax = plt.subplots(figsize=(12, 12))

# Plot Cairo boundary
cairo.plot(ax=ax, color='lightgrey', edgecolor='black', alpha=0.5)

# Plot the dissolved 500m road buffer
road_buffer_dissolved.plot(ax=ax, color='blue', alpha=0.3)

# Plot Cairo-only restaurants within the buffer
restaurants_near_clean.plot(ax=ax, color='red', markersize=5)

# Create custom legend handles
cairo_patch = mpatches.Patch(color='lightgrey', label='Cairo Boundary')
buffer_patch = mpatches.Patch(color='blue', alpha=0.3, label='500m Road Buffer')
restaurant_marker = mlines.Line2D([], [], color='red', marker='o', linestyle='None', markersize=5, label='Cairo Restaurants in Buffer')

plt.legend(handles=[cairo_patch, buffer_patch, restaurant_marker])

plt.title('Restaurants within 500m of Primary Roads (inside Cairo)')
ax.set_axis_off()
plt.show()

# Save the figure as a high-resolution PNG
#fig.savefig(os.path.join(output_folder, 'cariro restaurants_near_roads.png'), dpi=300, bbox_inches='tight')   # Uncomment to use